# MIC smoke test on ROSE (50 iterations)

Before running: Settings > Accelerator: **GPU T4 x1**, Internet: **on**.
Attach two datasets: the ROSE MIC data (contents of `data/mic/`) and `mit_b5.pth`.
Add-ons > Secrets: `GITHUB_TOKEN` (only needed if the repo is private).
Set `REPO` below to your GitHub repo.

In [ ]:
REPO = "YOUR_GITHUB_USER/weedrefine-uda"   # <- change this
PAIR_CFG = "configs/rose/bean_2019_to_2021_s0_smoke.py"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("weedrefine-uda"):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = f"https://{token}@github.com/{REPO}.git"
    except Exception:
        url = f"https://github.com/{REPO}.git"   # public repo
    subprocess.run(["git", "clone", "-q", url, "weedrefine-uda"], check=True)
    # remove the token from the saved git config so it is not in the notebook output
    subprocess.run(["git", "-C", "weedrefine-uda", "remote", "set-url", "origin",
                    f"https://github.com/{REPO}.git"], check=True)
os.chdir("/kaggle/working/weedrefine-uda")
!git log -1 --format='%h %s'

In [ ]:
# About 5 to 10 minutes. Ends with 'Setup done' and a config check line.
!bash scripts/kaggle_mic_setup.sh

In [ ]:
# 50 training iterations, then evaluation on the 25 target val images.
!cd third_party/MIC/seg && /root/miniforge/envs/mic/bin/python run_experiments.py --config {PAIR_CFG} 2>&1 | tail -n 80

In [ ]:
# Where the log went, peak GPU memory and the evaluation lines.
!ls -t third_party/MIC/seg/work_dirs/local-rose/ | head -3
!LOG=$(ls -t third_party/MIC/seg/work_dirs/local-rose/*/*.log | head -1); echo $LOG; grep -E 'memory|mIoU|weed|crop|background' $LOG | tail -n 25